WEEK 08 - THURSDAY FULL PIPELINE
Stock + Churn (Complete Assignment)

Author: Avishka Jindal

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# =========================
# CONFIG (NO MAGIC NUMBERS)
# =========================
WINDOW_SIZE = 20
TEST_RATIO = 0.2
EPOCHS = 10
BATCH_SIZE = 32


# =========================
# STEP 1: LOAD STOCK DATA
# =========================
def load_stock(file_path):
    try:
        df = pd.read_csv(file_path)
        print("Loaded stock data:", df.shape)
        return df
    except Exception as e:
        print("Error loading stock data:", e)
        return None


# =========================
# STEP 1B: PREPROCESS
# =========================
def preprocess_stock(df):
    df['date'] = pd.to_datetime(df['date'], format="%d-%m-%Y")
    df = df.sort_values('date')

    # choose one stock
    ticker = df['ticker'].unique()[0]
    df = df[df['ticker'] == ticker]

    print("Using stock:", ticker)

    return df[['date', 'close']]


# =========================
# STEP 1C: SEQUENCE BUILD
# =========================
def create_sequences(data, window):
    X, y = [], []

    for i in range(window, len(data)):
        X.append(data[i-window:i])
        y.append(data[i])

    return np.array(X), np.array(y)


# =========================
# STEP 1D: SPLIT (NO LEAKAGE)
# =========================
def split_time_series(data):
    split = int(len(data) * (1 - TEST_RATIO))
    return data[:split], data[split:]


# =========================
# STEP 3: LSTM MODEL
# =========================
def build_lstm(input_shape):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        LSTM(32),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model


# =========================
# STEP 6: BASELINE
# =========================
def baseline_predict(X):
    return np.array([x[-1] for x in X])


# =========================
# EVALUATION
# =========================
def evaluate(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)

    print(f"\n{name}")
    print("RMSE:", rmse)
    print("MAPE:", mape)


# =========================
# STOCK PIPELINE
# =========================
def run_stock_pipeline(file_path):
    df = load_stock(file_path)
    df = preprocess_stock(df)

    values = df['close'].values.reshape(-1, 1)

    # Split FIRST (important)
    train, test = split_time_series(values)

    scaler = MinMaxScaler()
    train_scaled = scaler.fit_transform(train)
    test_scaled = scaler.transform(test)

    # sequences
    X_train, y_train = create_sequences(train_scaled, WINDOW_SIZE)
    X_test, y_test = create_sequences(test_scaled, WINDOW_SIZE)

    # baseline
    baseline_preds = baseline_predict(X_test)
    evaluate(y_test, baseline_preds, "Baseline")

    # LSTM
    model = build_lstm((X_train.shape[1], 1))
    model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)

    lstm_preds = model.predict(X_test)
    evaluate(y_test, lstm_preds, "LSTM")

In [ ]:
# =========================
# STEP 2: LOAD CHAT DATA
# =========================
def load_chat(file_path):
    df = pd.read_csv(file_path)
    print("Loaded chat data:", df.shape)
    return df


# =========================
# STEP 2B: FIX TIMESTAMP
# =========================
def fix_timestamp(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df = df.dropna(subset=['timestamp'])
    return df


# =========================
# STEP 2C: FEATURE ENGINEERING
# =========================
def create_chat_features(df):
    df['message_length'] = df['message'].astype(str).apply(len)

    agg = df.groupby('customer_id').agg({
        'message_length': 'mean',
        'timestamp': 'count'
    }).rename(columns={'timestamp': 'msg_count'})

    # fake churn label (replace with real)
    agg['churn'] = (agg['msg_count'] < 5).astype(int)

    return agg.reset_index()


# =========================
# STEP 4: MODEL
# =========================
def train_churn_model(df):
    X = df[['message_length', 'msg_count']]
    y = df['churn']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

    model = RandomForestClassifier()
    model.fit(X_train, y_train)

    acc = model.score(X_test, y_test)
    print("\nChurn Model Accuracy:", acc)

    return model, X


# =========================
# STEP 5: RISK LIST
# =========================
def generate_risk_list(model, X):
    probs = model.predict_proba(X)[:, 1]

    df_out = X.copy()
    df_out['risk'] = probs

    df_out = df_out.sort_values('risk', ascending=False)

    print("\nTop 10 High-Risk Customers:")
    print(df_out.head(10))

In [ ]:
if __name__ == "__main__":
    # CHANGE PATHS ONLY HERE
    STOCK_PATH = "stock_prices.csv"
    CHAT_PATH = "chat_logs.csv"

    print("\n=== STOCK PIPELINE ===")
    run_stock_pipeline(STOCK_PATH)

    print("\n=== CHAT PIPELINE ===")
    chat_df = load_chat(CHAT_PATH)
    chat_df = fix_timestamp(chat_df)
    chat_features = create_chat_features(chat_df)

    model, X = train_churn_model(chat_features)
    generate_risk_list(model, X)